# Project: FinTech Subscription Optimization via A/B Testing

This notebook documents a complete A/B testing workflow for a premium subscription pricing page. It combines data generation, descriptive analytics, hypothesis testing, confidence estimation, and revenue impact modeling.

## Executive Summary

- Simulated 10,000 users in a randomized A/B experiment.
- **Control conversion:** 10.15%
- **Treatment conversion:** 11.89%
- **Absolute lift:** 1.74 percentage points
- **Relative lift:** 17.15%
- **P-value:** 0.0055 (statistically significant at the 0.05 level)
- **Projected annual revenue increase:** $870,233.15 on 1,000,000 visitors with $50 AOV.

## 1. Experimental Design & Randomization

This experiment uses reproducible group assignment and Bernoulli trials to model user conversion behavior:

- `np.random.seed(50)` ensures results can be recreated exactly.
- 10,000 users are randomly assigned to control or treatment.
- Control has baseline conversion probability of **10%**.
- Treatment has a higher planned conversion probability of **12%**.
- Conversion outcomes are generated using `np.random.binomial`.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

# 1. Experimental Parameters
users = 10000
control_rate = 0.10
treatment_rate = 0.12
np.random.seed(50)

# 2. Data Generation
group_labels = ['control', 'treatment']
groups = np.random.choice(group_labels, size=users)
probability = np.where(groups == 'control', control_rate, treatment_rate)
purchased = np.random.binomial(1, probability)

data = pd.DataFrame({
    'group': groups,
    'purchased': purchased
})

data['group'] = pd.Categorical(data['group'], categories=group_labels)

data.head()

## 2. Descriptive Statistics & Balance Check

Here we verify the experiment is balanced and compute the observed conversion metrics.

In [ ]:
conversion_summary = (
    data
    .groupby('group')
    .agg(users=('purchased', 'count'), conversions=('purchased', 'sum'))
    .assign(conversion_rate=lambda df: df['conversions'] / df['users'])
)

conversion_summary['relative_lift'] = (
    conversion_summary.loc['treatment', 'conversion_rate']
    - conversion_summary.loc['control', 'conversion_rate']
) / conversion_summary.loc['control', 'conversion_rate']

conversion_summary

In [ ]:
# Business impact assumptions
annual_visitors = 1_000_000
average_order_value = 50
absolute_lift = (
    conversion_summary.loc['treatment', 'conversion_rate']
    - conversion_summary.loc['control', 'conversion_rate']
)
expected_annual_gain = annual_visitors * absolute_lift * average_order_value

print("--- Experiment Balance ---")
print(conversion_summary[['users', 'conversions', 'conversion_rate']])
print()
print(f"Observed Absolute Lift: {absolute_lift:.4%}")
print(f"Observed Relative Lift: {conversion_summary.loc['treatment', 'relative_lift']:.2%}")
print(f"Expected Annual Revenue Increase: ${expected_annual_gain:,.2f}")

## 3. Hypothesis Testing & Confidence Interval

We test whether the treatment conversion rate is different from the control rate using a two-sample Z-test for proportions. This is appropriate for large samples and a binary outcome.

In [ ]:
control = data[data['group'] == 'control']['purchased']
treatment = data[data['group'] == 'treatment']['purchased']

successes = [int(control.sum()), int(treatment.sum())]
nobs = [int(control.count()), int(treatment.count())]

z_stat, p_value = proportions_ztest(count=successes, nobs=nobs)

se_diff = np.sqrt(
    conversion_summary.loc['control', 'conversion_rate'] * (1 - conversion_summary.loc['control', 'conversion_rate']) / nobs[0]
    + conversion_summary.loc['treatment', 'conversion_rate'] * (1 - conversion_summary.loc['treatment', 'conversion_rate']) / nobs[1]
)
ci_multiplier = 1.96
ci_lower = absolute_lift - ci_multiplier * se_diff
ci_upper = absolute_lift + ci_multiplier * se_diff

power_analysis = NormalIndPower()
effect_size = proportion_effectsize(
    conversion_summary.loc['control', 'conversion_rate'],
    conversion_summary.loc['treatment', 'conversion_rate']
)
power = power_analysis.power(
    effect_size=effect_size,
    nobs1=nobs[0],
    alpha=0.05,
    ratio=nobs[1] / nobs[0]
)

print(f"Control users: {nobs[0]}")
print(f"Treatment users: {nobs[1]}")
print(f"Z-score: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"95% CI for Absolute Lift: [{ci_lower:.4%}, {ci_upper:.4%}]")
print(f"Statistical power: {power:.3f}")
print(f"Significant at alpha=0.05? {p_value < 0.05}")


## 4. Financial Impact Scenario Analysis

This section translates the observed lift into revenue estimates under conservative, base-case, and aggressive assumptions.

In [ ]:
impact_scenarios = pd.DataFrame([
    {'scenario': 'conservative', 'traffic': 900_000, 'AOV': 45},
    {'scenario': 'base case', 'traffic': 1_000_000, 'AOV': 50},
    {'scenario': 'aggressive', 'traffic': 1_100_000, 'AOV': 55},
])
impact_scenarios['estimated_annual_gain'] = (
    impact_scenarios['traffic'] * absolute_lift * impact_scenarios['AOV']
)
impact_scenarios

## 5. Visual Evidence

The following plots make the result easy to interpret for stakeholders.

In [ ]:
sns.set_theme(style='whitegrid')

plt.figure(figsize=(10, 5))
aggregated = data.groupby('group')['purchased'].mean().reset_index()
ax = sns.barplot(
    x='group', y='purchased', data=aggregated,
    palette=['#B0C4DE', '#4682B4'], capsize=0.15
)
ax.set_title('Conversion Rate by Group', fontsize=16)
ax.set_xlabel('Experiment Group', fontsize=13)
ax.set_ylabel('Conversion Rate', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

for index, row in aggregated.iterrows():
    ax.text(index, row['purchased'] + 0.002, f"{row['purchased']:.2%}",
            ha='center', va='bottom', fontsize=11, fontweight='semibold')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.errorbar(
    ['Absolute Lift'], [absolute_lift],
    yerr=[[absolute_lift - ci_lower], [ci_upper - absolute_lift]],
    fmt='o', color='#4682B4', capsize=8, markersize=8
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Absolute Conversion Lift with 95% CI', fontsize=16)
plt.ylabel('Lift (percentage points)', fontsize=13)
plt.xticks([])
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.tight_layout()
plt.show()

## 5. Dataset Export

Export the simulated experiment dataset to `data/simulated_user_data.csv` so it can be reused for reporting or follow-up analysis.

In [ ]:
output_path = Path.cwd() / 'data' / 'simulated_user_data.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(output_path, index=False)
print(f"Saved simulated dataset to: {output_path}")


## 6. Key Findings and Recommendations

- The treatment page delivers a clear and statistically significant conversion lift.
- The result is strong enough to reject the null hypothesis with a P-value of 0.0055.
- The projected revenue impact is compelling for a rollout decision.

### Recommended next steps
1. Deploy the treatment page broadly while monitoring real conversion performance.
2. Validate the effect on retention and average order value after launch.
3. Run segment analysis by device, channel, and geography to identify where the lift is strongest.